MITOCHONDRIAL NETWORK ANALYSIS PIPELINE

1. Introduce variables

In [4]:
import subprocess
import os
import re
import shutil
import tifffile as tiff
from tifffile import imread, imwrite, imsave
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import igraph as ig
from tqdm.notebook import trange
import tifffile
from skimage import io, filters, morphology, measure
from skimage.filters import threshold_otsu, sobel
from scipy.ndimage import binary_dilation, binary_closing, binary_fill_holes, gaussian_filter, binary_erosion, generate_binary_structure
import mitotnt
import gc

RawNd2Path = "/home/jesusalarcon/Analysis/RawNd2Data"
NChannels = 1
TimeFrameInterval = 2   
LateralPixelSize = 111
ZDistance = 200
PSFZslices = 21
NumAperture = 1.42
RefractIndex = 1.515
NCores = 15
NIter = 20
MitoChannel = 583


TresholdMG = 500 #NLGF1, NLGF2, WT5, WT7
#TresholdMG = 600 #WT5
#TresholdMG = 800 #NLGF3, #NLGF5, #NLGF6
#TresholdMG = 900 #WT1, #WT3, #WT4
#TresholdMG = 1200 #NLGF7
#TresholdMG = 1300 #WT2
#TresholdMG = 1800 #NLGF4
#TresholdMG = 3200 #WT6

2. Convert nd2 to tiff

2.1. Crear la carpeta de conversion a tiff

In [5]:
RawNd2PathIndex =RawNd2Path.rfind('/')
RawTiffData = RawNd2Path[:(RawNd2PathIndex+1)] + "RawTiffData"
#os.makedirs(RawTiffData)

2.2 Crear la lista de las path de los archivos input y de los archivos output de nd2 a tiff

In [3]:
RawNd2Files = os.listdir(RawNd2Path)
RawNd2FilePaths = []

for i in range(len(RawNd2Files)):
    NameFileIndex = RawNd2Files[i].rfind('.')
    NameFile = RawNd2Files[i][:NameFileIndex]
    RawTiffFilePath = RawTiffData + "/" + str(NameFile)
    os.makedirs(RawTiffFilePath)
    RawNd2FilePath = RawNd2Path + "/" + str(RawNd2Files[i])
    RawNd2FilePaths.append(RawNd2FilePath)

RawTiffFiles = []

for file in RawNd2Files:
    TiffFile = file.replace(".nd2", "_T%t.tif")
    RawTiffFiles.append(TiffFile)

RawTiffFilePaths = []
RawTiffFolders = os.listdir(RawTiffData)

for i in range(len(RawTiffFiles)):
    for z in range(len(RawTiffFolders)):
        SameFileFolder = re.search(RawTiffFolders[z], RawTiffFiles[i])
        if SameFileFolder != None:
            RawTiffFilePath = RawTiffData + "/" + str(RawTiffFolders[z]) + "/" + str(RawTiffFiles[i])
            RawTiffFilePaths.append(RawTiffFilePath)

2.3 Convert the files

In [4]:
for i in range(len(RawNd2FilePaths)):
    for z in range(len(RawTiffFilePaths)):
        NameFolderIndex2 = RawTiffFilePaths[z].rfind('/')
        NameFolderIndex1 = RawTiffFilePaths[z][:NameFolderIndex2].rfind('/')
        NameFolder = RawTiffFilePaths[z][NameFolderIndex1+1:NameFolderIndex2]
        SameFileFolder = re.search(NameFolder, RawNd2FilePaths[i])
        if SameFileFolder != None:
            nd22tiff = ["bash", "-c", "bfconvert " + str(RawNd2FilePaths[i]) + str(" ") + str(RawTiffFilePaths[z])]
            subprocess.run(nd22tiff)

/home/jesusalarcon/Analysis/RawNd2Data/NLGF_astro_4mn_2um_0.5FPS_n6.nd2
ND2Reader initializing /home/jesusalarcon/Analysis/RawNd2Data/NLGF_astro_4mn_2um_0.5FPS_n6.nd2
Searching for blocks
Parsing block 'ND2 FILE SIG' 0%
Parsing block 'ImageMetadat' 0%
Parsing block 'ImageCalibra' 0%
Parsing block 'ImageMetadat' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 0%
Parsing block 'ImageDataSeq' 1%
Parsing block 'ImageDataSeq' 1%
Parsing block 'ImageDataSeq' 1%
Parsing block 'ImageDataSeq' 1%
Parsing block 'ImageDataSeq' 1%
Parsing block 'ImageDataSeq' 1%
Parsing bloc

3. Deconvolution with Deconwolf

3.1. Crear la carpeta y las carpetas de deconvolution

In [26]:
RawNd2PathIndex = RawNd2Path.rfind('/')
DeconTiffData = RawNd2Path[:(RawNd2PathIndex+1)] + "DeconTiffData"
os.makedirs(DeconTiffData)

EmWavesDeconTiffPaths = []

for i in range(len(RawNd2Files)):
    NameFileIndex = RawNd2Files[i].rfind('.')
    NameFile = RawNd2Files[i][:NameFileIndex]
    DeconTiffFilePath = DeconTiffData + "/" + str(NameFile)
    os.makedirs(DeconTiffFilePath)
    DeconTiffChannelData = DeconTiffFilePath + "/" + str(MitoChannel)
    os.makedirs(DeconTiffChannelData)
    EmWavesDeconTiffPaths.append(DeconTiffChannelData)

3.2 Crear los archivos PSF de los canales

In [27]:
PSFData = RawNd2Path[:(RawNd2PathIndex+1)] + "PSFData"
os.makedirs(PSFData)
RawTiffInputFolder = os.listdir(RawTiffData)

for channel in range(NChannels):
    PSFFile = PSFData + "/" + str(MitoChannel) + "PSF.tiff"
    createPSF = ["dw_bw", "--resxy", str(LateralPixelSize), "--resz", str(ZDistance), "--lambda", str(MitoChannel), "--NA", 
                                                                                                  str(NumAperture), "--ni",
                                                                                                  str(RefractIndex), "--nslice",
                                                                                                  str(PSFZslices), PSFFile]
    subprocess.run(createPSF)

Writing '/home/jesusalarcon/Analysis/PSFData/583PSF.tiff' (32-bit float)


3.3 Deconvolucionar los tiff files

In [28]:
RawTiffInputFolder = os.listdir(RawTiffData)

for y in range(len(RawTiffInputFolder)):
    RawTiffInputFiles = os.listdir(RawTiffData + "/" + RawTiffInputFolder[y])
    for i in range(len(RawTiffInputFiles)):
        DeconTiffChannelOutput = DeconTiffData + "/" + RawTiffInputFolder[y] + "/"  + str(MitoChannel) + "/" + RawTiffInputFiles[i]
        DeconTiffChannelInput = RawTiffData + "/" + RawTiffInputFolder[y] + "/" + RawTiffInputFiles[i]
        PSFFileInput = PSFData + "/" + str(MitoChannel) + "PSF.tiff"
        createDeconTiffFiles = ["dw", "--threads", str(NCores), "--iter", str(NIter), "--out", DeconTiffChannelOutput, DeconTiffChannelInput, PSFFileInput, "--gpu"]
        subprocess.run(createDeconTiffFiles)

TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T79.tif: Error fetching directory count.
TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T79.tif: Error fetching directory count.


Reading /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T79.tif
fim_tiff WARNING: Only WhiteIsZero or BlackIsZero are supported Photometric Interpretations, tag found: 3
Assuming min-is-black
Reading /home/jesusalarcon/Analysis/PSFData/583PSF.tiff
PSF XY-crop [181 x 181 x 21] -> [63 x 63 x 21]
Output: /home/jesusalarcon/Analysis/DeconTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/583/NLGF_astro_4mn_2um_0.5FPS_n6_T79.tif(.log.txt)
Deconvolving using shbcl2 (using inplace)
image: [1024x1024x11], psf: [63x63x21], job: [1086x1086x31]
Iteration  20/ 20, Idiv=7.890e-01            .....
scaling: 2.635106
Done!
Reading /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T35.tif
fim_tiff WARNING: Only WhiteIsZero or BlackIsZero are supported Photometric Interpretations, tag found: 3
Assuming min-is-black
Reading /home/jesusalarcon/Analysis/PSFData/583PSF.tiff
PSF XY-crop [181 x 181 x 21] -> [63 x 63 x 21]
Out

TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T35.tif: Error fetching directory count.
TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T35.tif: Error fetching directory count.


Iteration  20/ 20, Idiv=8.899e-01            .....
scaling: 2.389080
Done!
Reading /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T51.tif
fim_tiff WARNING: Only WhiteIsZero or BlackIsZero are supported Photometric Interpretations, tag found: 3
Assuming min-is-black


TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T51.tif: Error fetching directory count.
TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6_T51.tif: Error fetching directory count.


...

Iteration  20/ 20, Idiv=4.317e-01            .....
scaling: 12.520127
Done!
Reading /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T23.tif
fim_tiff WARNING: Only WhiteIsZero or BlackIsZero are supported Photometric Interpretations, tag found: 3
Assuming min-is-black


TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T23.tif: Error fetching directory count.
TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T23.tif: Error fetching directory count.


Reading /home/jesusalarcon/Analysis/PSFData/583PSF.tiff
PSF XY-crop [181 x 181 x 21] -> [63 x 63 x 21]
Output: /home/jesusalarcon/Analysis/DeconTiffData/WT_astro_4mn_2um_0.5FPS_n6/583/WT_astro_4mn_2um_0.5FPS_n6_T23.tif(.log.txt)
Deconvolving using shbcl2 (using inplace)
image: [1024x1024x11], psf: [63x63x21], job: [1086x1086x31]
Iteration  20/ 20, Idiv=4.248e-01            .....
scaling: 13.132529
Done!
Reading /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T46.tif
fim_tiff WARNING: Only WhiteIsZero or BlackIsZero are supported Photometric Interpretations, tag found: 3
Assuming min-is-black


TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T46.tif: Error fetching directory count.
TIFFAdvanceDirectory: tif_dir.c:1869: /home/jesusalarcon/Analysis/RawTiffData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6_T46.tif: Error fetching directory count.


Reading /home/jesusalarcon/Analysis/PSFData/583PSF.tiff
PSF XY-crop [181 x 181 x 21] -> [63 x 63 x 21]
Output: /home/jesusalarcon/Analysis/DeconTiffData/WT_astro_4mn_2um_0.5FPS_n6/583/WT_astro_4mn_2um_0.5FPS_n6_T46.tif(.log.txt)
Deconvolving using shbcl2 (using inplace)
image: [1024x1024x11], psf: [63x63x21], job: [1086x1086x31]
Iteration  20/ 20, Idiv=4.181e-01            .....
scaling: 14.503722
Done!


3.4 Move the log files to another folder

In [39]:
LogFile = '.log.txt'
DeconLogFolder = RawNd2Path[:(RawNd2PathIndex+1)] + "/" + "DeconLog"
os.makedirs(DeconLogFolder)

DeconTiffInputFolder = os.listdir(DeconTiffData)

for y in range(len(DeconTiffInputFolder)):
    DeconTiffInputFiles = os.listdir(DeconTiffData + "/" + DeconTiffInputFolder[y])
    DeconTiffFolder = DeconTiffData + "/" + DeconTiffInputFolder[y] + "/" + str(MitoChannel)
    DeconTiffFiles = os.listdir(DeconTiffFolder)
    for x in range(len(DeconTiffFiles)):
        DeconLogFile = re.search(LogFile, DeconTiffFiles[x])
        if DeconLogFile != None:
            DeconLogPath = DeconTiffFolder + "/" + DeconTiffFiles[x]
            DeconLogFolderPath = DeconLogFolder + "/" + DeconTiffFiles[x]
            shutil.move(DeconLogPath, DeconLogFolder)

4. MitoTNT

4.1 Create MitoTNT input folder

In [129]:
DeconTiffData = RawNd2Path[:(RawNd2PathIndex+1)] + "DeconTiffData"
DeconTiffInputFolder = os.listdir(DeconTiffData)
RawNd2Files = os.listdir(RawNd2Path)
MitoGraphFolder = RawNd2Path[:(RawNd2PathIndex+1)] + "MitoGraphData"
#os.makedirs(MitoGraphFolder)

RawNd2Files = ["WT_astro_4mn_2um_0.5FPS_n7.nd2"]

for i in range(len(RawNd2Files)):
    NameFileIndex = RawNd2Files[i].rfind('.')
    NameFile = RawNd2Files[i][:NameFileIndex]
    MitoGraphFilePath = MitoGraphFolder + "/" + str(NameFile) + "/" + str(NameFile)
    os.makedirs(MitoGraphFilePath)

MitoGraphFolders = os.listdir(MitoGraphFolder)

MitoGraphFolders = ["WT_astro_4mn_2um_0.5FPS_n7"]

for z in range(len(MitoGraphFolders)):
    DeconTiffMitoFolder = DeconTiffData + "/" + str(MitoGraphFolders[z]) + "/" + str(MitoChannel)
    DeconTiffMitoFiles = os.listdir(DeconTiffMitoFolder)
    for i in range(len(DeconTiffMitoFiles)):
        DeconTiffMitoFileParts = DeconTiffMitoFiles[i].split('.tif')
        DeconTiffMitoFileIndex = DeconTiffMitoFileParts[0].rfind('T')
        DeconTiffMitoFileFrame = DeconTiffMitoFileParts[0][DeconTiffMitoFileIndex+1:len(DeconTiffMitoFileParts[0])]
        if int(DeconTiffMitoFileFrame) < 60:
            MitoGraphTimeFolder = MitoGraphFolder + "/" + str(MitoGraphFolders[z]) + "/" + str(MitoGraphFolders[z]) + "/" + "frame_" + str(DeconTiffMitoFileFrame)
            os.makedirs(MitoGraphTimeFolder)

DeconMitoInputFolder = os.listdir(DeconTiffData)

DeconMitoInputFolder = ["WT_astro_4mn_2um_0.5FPS_n7"]
DeconTiffInputFolder = ["WT_astro_4mn_2um_0.5FPS_n7"]

for y in range(len(DeconMitoInputFolder)):
    DeconMitoInputPath = DeconTiffData + "/" + str(DeconTiffInputFolder[y]) + "/" + str(MitoChannel)
    DeconMitoInputData = os.listdir(DeconMitoInputPath)
    for i in range(len(DeconMitoInputData)):
        DeconTiffMitoFilePath = DeconMitoInputPath + "/" + DeconMitoInputData[i]
        DeconTiffMitoStack = imread(DeconTiffMitoFilePath)

        DeconTiffMitoStackNorm2 = DeconTiffMitoStack / TresholdMG
        DeconTiffMitoStackNorm2[DeconTiffMitoStackNorm2>1.0] = 255
        #DeconTiffMitoStackNorm3 = DeconTiffMitoStackNorm2 * 255

        DeconTiffMitoStackNormSelec = DeconTiffMitoStackNorm2[:, 0:1024, 0:1024]

        DeconTiffMitoStackNormSelec = DeconTiffMitoStackNormSelec.astype(np.uint8)

        MitoGraphFilesFolders = MitoGraphFolder + "/" + str(DeconTiffInputFolder[y]) + "/" + str(DeconTiffInputFolder[y])
        MitoGraphFrameFolders = os.listdir(MitoGraphFilesFolders)
        
        for z in range(len(MitoGraphFrameFolders)):
            DeconTiffMitoFileParts = DeconMitoInputData[i].split('.tif')
            DeconTiffMitoFileIndex = DeconTiffMitoFileParts[0].rfind('T')
            DeconTiffMitoFileFrame = DeconTiffMitoFileParts[0][DeconTiffMitoFileIndex+1:len(DeconTiffMitoFileParts[0])]
            DeconTiffMitoFileCompleteFrame = "frame_" + str(DeconTiffMitoFileFrame)
            if int(DeconTiffMitoFileFrame) < 60:
                if DeconTiffMitoFileCompleteFrame == MitoGraphFrameFolders[z]:
                    SaveMitoGraphFileFolder = MitoGraphFolder + "/" + str(DeconTiffInputFolder[y]) + "/" + str(DeconTiffInputFolder[y]) + "/" + str(MitoGraphFrameFolders[z]) + "/" + DeconTiffMitoFileCompleteFrame + ".tif"
                    imwrite(SaveMitoGraphFileFolder, DeconTiffMitoStackNormSelec)

<tifffile.TiffFile 'WT_astro_4mn_2um…5FPS_n7_T96.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2um…5FPS_n7_T60.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2u…5FPS_n7_T107.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2u…5FPS_n7_T111.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2u…5FPS_n7_T101.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2u…5FPS_n7_T114.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2um…5FPS_n7_T50.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2um…5FPS_n7_T92.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2u…0.5FPS_n7_T3.tif'> ImageJ series metadata invalid or corrupted file
<tifffile.TiffFile 'WT_astro_4mn_2um…5FPS_n7_T19.tif'> ImageJ se

4.2. MitoTNT preprocessing

In [3]:
MitoGraphFolder = RawNd2Path[:(RawNd2PathIndex+1)] + "MitoGraphData"
MitoGraphFolderSamples = os.listdir(MitoGraphFolder)
xysize = LateralPixelSize/1000
zsize = ZDistance/1000
mitopattern = "frame"    

for x in range(len(MitoGraphFolderSamples)):
    MitoGraphFile2AnalyzePath = str(MitoGraphFolder) + "/" + str(MitoGraphFolderSamples[x]) + "/" + str(MitoGraphFolderSamples[x])
    seg = mitotnt.MitoSegmenter(MitoGraphFile2AnalyzePath, mitopattern, xysize, zsize)
    seg.run_mitograph_cpu(max_workers=NCores, extra_params = " -analyze")
    seg.visualize_segmented_data()

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_47' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_37' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_12' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_13' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_8' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_0' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_58' has already been processed
'/home/jesusalarcon/Analysis/MitoGra

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_13' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_11' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_17' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_15' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_2' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_18' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_1' has already been processed
'/home/jesusalarcon/Analysis/MitoGra

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

...

[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n5/NLGF_astro_4mn_2um_0.5FPS_n5/frame_20/frame_20	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n5/NLGF_astro_4mn_2um_0.5FPS_n5/frame_14/frame_14	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n5/NLGF_astro_4mn_2um_0.5FPS_n5/frame_16/frame_16	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n5/NLGF_astro_4mn_2um_0.5FPS_n5/frame_13/frame_13	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n4/NLGF_astro_4mn_2um_0.5FPS_n4/frame_15/frame_15	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n4/NLGF_astro_4mn_2um_0.5FPS_n4/frame_19/frame_19	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n4/NLGF_astro_4mn_2um_0.5FPS_n4/frame_12/frame_12	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n4/NLGF_astro_4mn_2um_0.5FPS_n4/frame_0/frame_0	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/An

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_16/frame_16	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_0/frame_0	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_15/frame_15	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_20/frame_20	[done]
Process complete. Exiting MitoGraph...
[Starting MitoGraph v3.1]
1 files found. MitoGraph is running...
/home/jesusalarcon/Analysis/MitoGraph

In [6]:
MitoGraphFolder = RawNd2Path[:(RawNd2PathIndex+1)] + "MitoGraphData"
MitoGraphFolderSamples = os.listdir(MitoGraphFolder)
xysize = LateralPixelSize/1000
zsize = ZDistance/1000
mitopattern = "frame"    

for x in range(len(MitoGraphFolderSamples)):
    MitoGraphFile2AnalyzePath = str(MitoGraphFolder) + "/" + str(MitoGraphFolderSamples[x]) + "/" + str(MitoGraphFolderSamples[x])
    seg = mitotnt.MitoSegmenter(MitoGraphFile2AnalyzePath, mitopattern, xysize, zsize)
    seg.run_mitograph_cpu(max_workers=NCores, extra_params = " -analyze")
    seg.visualize_segmented_data()
    skel = mitotnt.SkeletonizedMito(seg)
    skel.extract_graphs(node_gap_size = 2)

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_4' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_26' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_29' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_24' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_10' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_56' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n6/NLGF_astro_4mn_2um_0.5FPS_n6/frame_21' has already been processed
'/home/jesusalarcon/Analysis/MitoGr

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_26' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_58' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_48' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_41' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_32' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_37' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/NLGF_astro_4mn_2um_0.5FPS_n7/NLGF_astro_4mn_2um_0.5FPS_n7/frame_28' has already been processed
'/home/jesusalarcon/Analysis/MitoG

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

...

Running MitoGraph:   0%|          | 0/60 [00:00<?, ?it/s]

'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_3' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_2' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_24' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_0' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_9' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_48' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS_n6/WT_astro_4mn_2um_0.5FPS_n6/frame_10' has already been processed
'/home/jesusalarcon/Analysis/MitoGraphData/WT_astro_4mn_2um_0.5FPS

Extracting full graphs and segment nodes:   0%|          | 0/60 [00:00<?, ?it/s]

Extracting simple graphs:   0%|          | 0/60 [00:00<?, ?it/s]

Extracting local simple graphs:   0%|          | 0/60 [00:00<?, ?it/s]

In [130]:
MitoGraphFolder = RawNd2Path[:(RawNd2PathIndex+1)] + "MitoGraphData"
MitoGraphFolderSamples = os.listdir(MitoGraphFolder)
xysize = LateralPixelSize/1000
zsize = ZDistance/1000
mitopattern = "frame"    

for x in range(len(MitoGraphFolderSamples)):
    MitoGraphFile2AnalyzePath = str(MitoGraphFolder) + "/" + str(MitoGraphFolderSamples[x]) + "/" + str(MitoGraphFolderSamples[x])
    seg = mitotnt.MitoSegmenter(MitoGraphFile2AnalyzePath, mitopattern, xysize, zsize)
    seg.run_mitograph_cpu(max_workers=NCores, extra_params = " -analyze")
    seg.visualize_segmented_data()
    skel = mitotnt.SkeletonizedMito(seg)
    skel.extract_graphs(node_gap_size = 2)
    del seg
    tracker = mitotnt.NetworkTracker(skel, frame_interval=TimeFrameInterval, min_track_size = 2, max_gap_size = 1)
    tracked = tracker.run()
    del skel
    viz = mitotnt.Visualizer(tracked)
    viz.generate_chimerax_skeleton()
    viz.generate_tracking_arrows()
    viz.visualize_tracking()
    tracked.detect_fusion_fission()
    tracked.compute_mitochondria_diffusivity()
    del tracker, tracked, viz
    gc.collect()


KeyboardInterrupt


KeyboardInterrupt

